In [1]:
# Copyright IonQ Inc., all rights reserved
import numpy as np
from pickle import dump

import analysis
import dem_decode

from replace_noise import replace_all_noise as rep_all
from multithread_decode import decode_all
decode_constructor = dem_decode.decoder_constructor

In [ ]:
# Get Ion -> Qubit Maps, and DRB data
ion_drb = analysis.build_ion_drb_lut('SpanningDRB_all.csv')
# qb_map  = analysis.read_qubit_map('qbmap_other.dat') # <---- Use this map for Toric + Concatenated + 422 codes
qb_map  = analysis.read_qubit_map('qbmap_qldpc.dat') # <---- Use this map for BB + GB/ codes


In [ ]:
""" Read raw data + decoding info. Dictionary structure is:
    'circuit name': (
        <Initialized decoder object>,
        NumPy array of syndrome detector bits (XOR'ed as appropriate),
        NumPy array of measured logical values,
        Stim Circuit.
    )
"""
# decoding_dict = analysis.read_decoding_dict('intermediate_data/Mar2/Other.dat', decode_constructor)
decoding_dict = analysis.read_decoding_dict('intermediate_data/Mar2/Qldpc.dat', decode_constructor)

In [4]:
# (OPTIONALLY) Fine-tune noise model, regenerate DEM + reinitialize decoder where needed.
layer_drb_mean_scale = 0.011
for key in decoding_dict.keys():
    shortkey = key.split("/")[-1].replace(".conf.js","")
    qb_drb_lut = analysis.build_qubit_drb_map(ion_drb, *qb_map[shortkey])
    µ_drb_lut = np.mean(list(qb_drb_lut.values()))

    for key2 in qb_drb_lut.keys():
        qb_drb_lut[key2] = qb_drb_lut[key2] * layer_drb_mean_scale / µ_drb_lut

    newcirc = rep_all(
        decoding_dict[key][3], qb_drb_lut,
        1.0e-5 * 4.28,
        1.0e-5,
        1.1e-4 * 4.28,
        1.1e-4,
        3.2e-4,
        0.008

    )
    newdem = newcirc.detector_error_model()
    newdec = decode_constructor(newdem)
    dets = decoding_dict[key][1]
    logs = decoding_dict[key][2]
    decoding_dict[key] = (newdec, dets, logs, newcirc)

In [5]:
# Call decoder, then print results
result_dict = decode_all(decoding_dict, processes=8)

In [6]:
# Quickly print results as sneak-peak. Use Plot_Data.ipynb to plot properly.
decref = []
processed_results = {}
print(f"{'Exp. Name':29s} | {'Raw LER':7s} | Decoded LER")
# print(f"                     Prog name:                |  Raw: | Dec:")
for key in result_dict.keys():
    splitkey = key.split("/")
    experiment_name = splitkey[-1]

    stats = result_dict[key]
    shot = len(stats) # Num. of shots
    rnd = int(experiment_name.split('_')[2][1:]) # Num. of rounds

    raw = np.sum([s[0] for s in stats])/shot/rnd
    dec = np.sum([s[1] for s in stats])/shot/rnd
    
    print(f"{experiment_name:30s}: {100*raw:6.2f}% | {100*dec:6.2f}%")
    decref.append(dec)
    processed_results[key] = 100*dec

Exp. Name                     | Raw LER | Decoded LER
BB5final_d3_r6_X0             :  11.87% |   5.54%
BB5final_d3_r6_X1             :  12.08% |   5.83%
BB5final_d3_r6_Z0             :  10.79% |   3.81%
BB5final_d3_r6_Z1             :  10.98% |   3.87%
BB5final_d4_r6_X0             :  12.77% |   6.86%
BB5final_d4_r6_X1             :  13.22% |   6.76%
BB5final_d4_r6_Z0             :  13.07% |   4.20%
BB5final_d4_r6_Z1             :  12.85% |   4.07%
BB5final_d5_r6_X0             :  14.91% |   9.44%
BB5final_d5_r6_X1             :  14.98% |   9.58%
BB5final_d5_r6_Z0             :  14.22% |   6.08%
BB5final_d5_r6_Z1             :  14.33% |   5.24%
GB4final_d4_r6_X0             :   8.64% |   2.68%
GB4final_d4_r6_X1             :   8.92% |   2.58%
GB4final_d4_r6_Z0             :   7.82% |   2.01%
GB4final_d4_r6_Z1             :   7.66% |   1.75%
GB4final_d5_r6_X0             :  11.83% |   3.44%
GB4final_d5_r6_X1             :  11.61% |   3.28%
GB4final_d5_r6_Z0             :  10.11% |   1.

In [ ]:
# (OPTIONALLY) Export to file, for plotting etc.
# with open("processed_data/Mar2_QLDPC_Beam32.dat", "wb") as f:
#     dump(result_dict, f)